In [ ]:
import os
from pathlib import Path

import numpy as np
import torch as tc
import torchvision.transforms.v2 as tvs
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from tqdm import tqdm

from convnet import ConvNet
# from scatnet import ScatNet

RNG = tc.Generator().manual_seed(0)

# FIXME: remove transforms from validation and testing subsets
transforms = tvs.Compose([
    tvs.Grayscale(),
    tvs.ToImage(),
    # tvs.CenterCrop(128),
    tvs.Resize((128, 128)),
    # bit of data augmentation
    tvs.RandomHorizontalFlip(),
    tvs.GaussianNoise(),
    tvs.ConvertImageDtype(tc.float32),
])

ROOT = Path(os.environ["CHEST_XRAY"])

dataset = tc.utils.data.ConcatDataset([
    ImageFolder(Path(ROOT, "train"), transform=transforms),
    ImageFolder(Path(ROOT, "test"), transform=transforms),
])

BATCH_SIZE: int = 100
WORKERS: int = 4

devel_dataset, test_dataset = tc.utils.data.random_split(dataset, [0.8, 0.2], generator=RNG)

device = tc.device("cuda")

SHAPE = (1, 128, 128)
model = ConvNet(shape=SHAPE).to(device)
# model.compile()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

EPOCHS: int = 50

criterion = tc.nn.CrossEntropyLoss().to(device)
optimizer = tc.optim.Adam(model.parameters())

acc_train = []; f1_train = []; loss_train = []
acc_valid = []; f1_valid = []; loss_valid = []

train_subset, valid_subset = tc.utils.data.random_split(devel_dataset, [0.8, 0.2], generator=RNG)
train_loader = DataLoader(
    train_subset,
    shuffle=True,
    batch_size=BATCH_SIZE,
    num_workers=WORKERS,
    pin_memory=True,
)
valid_loader = DataLoader(
    valid_subset,
    shuffle=True,
    batch_size=BATCH_SIZE,
    num_workers=WORKERS,
    pin_memory=True,
)

# for epoch in tqdm(range(EPOCHS), "Training"):
for epoch in range(EPOCHS):
    print(f"Epoch: {epoch}")

    running_loss = []
    running_acc = []
    running_f1 = []
    model.train()
    for images, labels in tqdm(train_loader, "Train on batches"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Compute accuracy, and F1-score
        predicted = tc.argmax(outputs, dim=-1)
        running_loss.append(loss.item())
        running_acc.append(accuracy_score(labels.cpu(), predicted.cpu()))
        running_f1.append(f1_score(labels.cpu(), predicted.cpu(), average="weighted"))
        # print("Batch accuracy score", accuracy_score(labels.cpu(), predicted.cpu()))

    loss_train.append(np.mean(running_loss))
    acc_train.append(np.mean(running_acc))
    f1_train.append(np.mean(running_f1))

    print(f"Train - Loss: {loss_train[epoch]:.4f}, Acc: {acc_train[epoch]:.4f}, F1: {f1_train[epoch]:.4f}")

    # Validation phase
    model.eval()
    running_loss = []
    running_acc = []
    running_f1 = []
    with tc.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            _, predicted = tc.max(outputs, 1)
            running_loss.append(loss.item())
            running_acc.append(accuracy_score(labels.cpu(), predicted.cpu()))
            running_f1.append(
                f1_score(labels.cpu(), predicted.cpu(), average="weighted")
            )

        loss_valid.append(np.mean(running_loss))
        acc_valid.append(np.mean(running_acc))
        f1_valid.append(np.mean(running_f1))

        print(f"Valid - Loss: {loss_valid[epoch]:.4f}, Acc: {acc_valid[epoch]:.4f}, F1: {f1_valid[epoch]:.4f}")

In [ ]:
# test the model

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=True,
)

def test_model(model: tc.nn.Module, images_loader: DataLoader):
    model.eval()
    all_preds = []
    all_labels = []
    with tc.no_grad():
        for images, labels in images_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            predicted = tc.argmax(outputs, dim=-1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    print(f'Test Accuracy: {accuracy:.4f}, Test F1-score: {f1:.4f}')

test_model(model, test_loader)

In [ ]:
# visualize filters
import matplotlib.pyplot as plt
import torchvision

for i, layer in enumerate([model.conv0.weight, model.conv1.weight]):
    print(layer.shape)
    fig, ax = plt.subplots()
    ax.set_title(f"conv layer {i}")
    image = torchvision.utils.make_grid(layer.cpu(), nrow=20, normalize=True, scale_each=True)
    # bring channel axis to the last place
    ax.imshow(image.numpy().transpose((1, 2, 0)))
    ax.axis("off")
    fig.tight_layout()

In [ ]:
# export the trained model

save_folder = Path("./weights")
save_folder.mkdir(exist_ok=True)
tc.save(model.state_dict(), Path(save_folder, f"{model.name}.pt"))